# 02 - Exploratory Data Analysis (EDA)

[Back to the ETL notebook](01_etl.ipynb)

This notebook explores the cleaned HMRC data to answer our business questions: which taxes matter most, how receipts change through the year, and how they have moved since 2017.

In [1]:
# Cell 1
# Bring in pandas, our tool for working with tables
import pandas as pd

# Load the cleaned file we saved in the ETL notebook
# parse_dates makes sure Month comes back as real dates
df = pd.read_csv("../data/processed/hmrc_clean.csv", parse_dates=["Month"])

# Use Month as the row label, which makes working with time much easier later
df = df.set_index("Month")

# Check the size, which should now be (113, 44) because Month became the index
print(df.shape)

(113, 44)


**Cell 1:**

**Action:** Loaded the cleaned HMRC data and set `Month` as the index, so every row is labelled by its date.

**Result:** 113 rows and 44 columns. Month is now the row label instead of a column, which is why the column count dropped from 45 to 44.

In [2]:
# Cell 2
# The biggest and most important taxes, to look at first
main_taxes = ["Total HMRC Receipts", "Income Tax", "National Insurance Contributions",
              "Value Added Tax", "Corporation Tax"]

# describe() gives the count, mean, spread (std), min, quartiles and max
# The 50% row is the median. Everything is in £ million, rounded to whole numbers
df[main_taxes].describe().round(0)

,Total HMRC Receipts,Income Tax,National Insurance Contributions,Value Added Tax,Corporation Tax
count,113.0,113.0,113.0,113.0,113.0
mean,61517.0,20123.0,13542.0,12442.0,5882.0
std,16666.0,7702.0,2335.0,4378.0,4076.0
min,26023.0,11001.0,9425.0,-896.0,904.0
25%,50272.0,14500.0,11566.0,9162.0,2453.0
50%,59566.0,18765.0,13496.0,12850.0,4314.0
75%,68886.0,22699.0,14685.0,15230.0,8375.0
max,126444.0,51255.0,20918.0,21753.0,18322.0


**Cell 2**

**Action:** Calculated summary statistics for the total receipts and the four biggest taxes.

**Result:** Across the 113 months, total receipts averaged about £61.5bn a month, with a median of about £59.6bn. The spread (standard deviation) is about £16.7bn, so receipts swing a lot from month to month. The lowest month was about £26.0bn and the highest about £126.4bn.

**Note:** Value Added Tax has a minimum below zero, meaning that in at least one month refunds were larger than payments. We'll look at that in the charts.

In [ ]:
# Cell 3
# Leave out the total columns and the sub-categories, so nothing is counted twice
exclude = ["Total Paid Over", "Total HMRC Receipts",
           "PAYE NIC1 (EMP'er)", "PAYE NIC1 (EMP'ee)", "SA NIC2&4"]
exclude += [c for c in df.columns if "included within" in c]

# Work out the average monthly receipts for each remaining tax, biggest first
avg_by_tax = df.drop(columns=exclude).mean().sort_values(ascending=False).round(0)

# Show the top 8
print(avg_by_tax.head(8))

Income Tax                          20123.0
National Insurance Contributions    13542.0
Value Added Tax                     12442.0
Corporation Tax                      5882.0
Hydrocarbon Oil (Fuel duties)        2125.0
Capital Gains Tax                    1093.0
Stamp Duty Land Tax                  1075.0
Energy Profits Levy                   954.0
dtype: float64


**Cell 3**

**Action:** Ranked the taxes by their average monthly receipts, leaving out totals and sub-categories so nothing was counted twice.

**Result:** Income Tax is the biggest at about £20.1bn a month, then National Insurance (£13.5bn), VAT (£12.4bn) and Corporation Tax (£5.9bn). Together those four make up most of what HMRC collects. Fuel duties come fifth at about £2.1bn.

In [4]:
# Cell 4
# Pull out the total receipts column to keep the lines below short
total = df["Total HMRC Receipts"]

# Find the month with the biggest total and the month with the smallest
print("Highest month:", total.idxmax().strftime("%B %Y"), "at", total.max())
print("Lowest month:", total.idxmin().strftime("%B %Y"), "at", total.min())

Highest month: January 2026 at 126444.0
Lowest month: May 2020 at 26023.0


**Cell 4**

**Action:** Found the months with the highest and lowest total receipts.

**Result:** The highest month was January 2026 at about £126.4bn and the lowest was May 2020 at about £26.0bn.

**Note:** January is likely high because many Self Assessment tax bills are due that month. May 2020 falls in the first Covid lockdown, which probably explains the dip. We'll check both patterns in the charts.